# EDA: House Listings, Kings County

# Question to answer

Hypothesis that I want to check:

- Houses in the city cost more than the suburbs
- Bigger the house, more expensive it is
- Houses in central city are more expensive than other areas in the city

**My client**
*William Rodriguez* 
- Buyer 
- 2 people, 
- country (best timing & non-renovated) & 
- city house (fast & central location), (wants two houses)

# 1. Initial Analysis

### 1. Importing all modules and setiing things up**

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import missingno as msno
from io import StringIO # creates an in-memory file-like object that behaves like a file, but stores its contents in a string instead of on disk.
from scipy import stats # for stat analysis
from matplotlib.ticker import PercentFormatter   # Import formatter to display axis values as percentages
from scipy.cluster import hierarchy
import folium
from folium import plugins
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time
from folium.plugins import MarkerCluster


# Update default Matplotlib settings for consistent plot appearance
plt.rcParams.update({
    "figure.figsize": (8, 5),      # Set default figure size
    "axes.facecolor": "white",     # Set plot background color
    "axes.edgecolor": "black"      # Set border color of the plot area
})

plt.rcParams["figure.facecolor"] = "w"            # Set overall figure background to white

pd.plotting.register_matplotlib_converters()      # Enable pandas datetime converters for Matplotlib plots

# Set pandas option to format floating-point numbers to 3 decimal places when displayed
pd.set_option('display.float_format', lambda x: '%.3f' % x)


In [ ]:
print("\n## 1. Initial Data Inspection")
print("\n")

# importing data
df_eda = pd.read_csv('../eda.csv')
print(df_eda.head(5))

# make a copy of the file for further steps
# Make a copy of the DataFrame
df1 = df_eda.copy()


## 1. Initial Data Inspection


         date      price          id  bedrooms  bathrooms  sqft_living  \
0  2014-10-13 221900.000  7129300520     3.000      1.000     1180.000   
1  2014-12-09 538000.000  6414100192     3.000      2.250     2570.000   
2  2015-02-25 180000.000  5631500400     2.000      1.000      770.000   
3  2014-12-09 604000.000  2487200875     4.000      3.000     1960.000   
4  2015-02-18 510000.000  1954400510     3.000      2.000     1680.000   

   sqft_lot  floors  waterfront  view  ...  grade  sqft_above  sqft_basement  \
0  5650.000   1.000         NaN 0.000  ...      7    1180.000          0.000   
1  7242.000   2.000       0.000 0.000  ...      7    2170.000        400.000   
2 10000.000   1.000       0.000 0.000  ...      6     770.000          0.000   
3  5000.000   1.000       0.000 0.000  ...      7    1050.000        910.000   
4  8080.000   1.000       0.000 0.000  ...      8    1680.000          0.000   

   yr_built  yr_renovated  zipcode    lat

In [ ]:
print(f"Dataset shape: {df1.shape}")
print(f"Number of properties: {len(df1)}")
print(f"Date range: {df1['date'].min()} to {df1['date'].max()}")
print("\n")

print("=== DATA TYPES AND BASIC INFO ===")
# check for data types and statistics overview of data frame columns
print(df1.info())

# check data types in data frame
df1.dtypes

### 2. Data Cleaning and Preparation
- Identify columns and adjust them
- Missing value imputation

In [ ]:
print("\n## 2. Data Cleaning and Preparation")

# Step 1: Identify columns with missing values (NaN or empty string in this case).
    # The initial df.info() showed non-null counts, let's look for empty strings/NaN explicitly.
    # We'll replace empty strings with NaN for proper imputation/handling.
df1 = df1.replace('', np.nan)

# Check missing values BEFORE imputation
print("=== MISSING VALUES BEFORE IMPUTATION ===")

# display number of missing values per column
    ## df.isna(): This returns a DataFrame of the same shape as df, but with boolean values:
    ## True → the cell is missing (NaN, None, or NaT)
    ## False → the cell is not missing
print(df1.isnull().sum())
print("\n")

df1.isna().sum()  * 100 / len(df1) # * 100 / len(df) gives the percentage of data missing

# For numerical columns (sqft_basement, yr_renovated):
    ## Use median or mode imputation
    ## For yr_renovated, 0 typically means "never renovated"

# For categorical/ordinal columns (waterfront, view):
    ## Use mode (most frequent value) imputation

    ## For waterfront, likely impute with 0 (not waterfront)
    ## For view, impute with 0 (no view)


## 2. Data Cleaning and Preparation
=== MISSING VALUES BEFORE IMPUTATION ===
date                0
price               0
id                  0
bedrooms            0
bathrooms           0
sqft_living         0
sqft_lot            0
floors              0
waterfront       2391
view               63
condition           0
grade               0
sqft_above          0
sqft_basement     452
yr_built            0
yr_renovated     3848
zipcode             0
lat                 0
long                0
sqft_living15       0
sqft_lot15          0
dtype: int64




date             0.000
price            0.000
id               0.000
bedrooms         0.000
bathrooms        0.000
sqft_living      0.000
sqft_lot         0.000
floors           0.000
waterfront      11.071
view             0.292
condition        0.000
grade            0.000
sqft_above       0.000
sqft_basement    2.093
yr_built         0.000
yr_renovated    17.817
zipcode          0.000
lat              0.000
long             0.000
sqft_living15    0.000
sqft_lot15       0.000
dtype: float64

*Why Imputation over Interpolation:*

Non-temporal data: Your dataset isn't time-series data where interpolation makes sense. The dates are sale dates, not sequential measurements of the same property.

Categorical nature: Some missing fields like waterfront (likely 0/1) and view (ordinal scale 0-4) are categorical/ordinal, not continuous.

Small dataset: With only 25 records and few missing values, interpolation could introduce unnecessary complexity.

Missing pattern: The missing values are sporadic across different records and columns, not following a pattern that interpolation would help.